In [ ]:
// load the formatting setup and the style sheet
#load "../setup.fsx"
HTML(System.IO.File.ReadAllText "../style.css")

# Infinite Sequences and the Sieve of Eratosthenes

TypeScript has *generators*, i.e. functions that produce a stream of values one at a time.  The F$^\sharp$ counterpart of a generator is a *sequence expression*: The expression `seq { ... }` creates a value of type `seq<int>`, i.e. a sequence of integers.  Inside the curly braces, the keyword `yield` produces the next element of the sequence.  The elements of a sequence are computed *lazily*: An element is only computed when it is needed.  Therefore, a sequence can be infinite.

The function `countFrom` takes one argument:
- `n` is an integer.

It returns the following *infinite stream* of natural numbers:
$$ [k \in \mathbb{N} \mid n \leq k] = [n,\; n+1,\; n+2,\; \cdots]$$
The mutable variable `k` stores the number that is produced next.  As the condition of the `while` loop is always `true`, the sequence never ends.

In [ ]:
let countFrom (n: int) : seq<int> =
    seq { let mutable k = n
          while true do
              yield k
              k <- k + 1 }

In [ ]:
let natNums = countFrom 0

When a sequence is displayed, only its first four elements are computed.

In [ ]:
natNums

Unlike a TypeScript generator, an F$^\sharp$ sequence has no state: Every time a sequence is traversed, its elements are computed anew, starting with the first one.  In order to fetch the elements of a sequence one at a time, we need an *enumerator*.  The method `GetEnumerator()` of a sequence creates an enumerator, which has the following members:
  * `MoveNext()` advances the enumerator to the next element and returns `true` if there is such an element,
  * `Current` is the element that the enumerator is positioned at.

The function `next` takes one argument:
- `e` is an enumerator of integers that has at least one more element.

It advances `e` and returns the next element of `e`.  The enumerator `e` is changed in the process.  The value of `MoveNext()` is ignored, since the sequences in this notebook are infinite.

In [ ]:
open System.Collections.Generic

let next (e: IEnumerator<int>) : int =
    e.MoveNext() |> ignore
    e.Current

In [ ]:
let natNumsEnum = natNums.GetEnumerator()

In [ ]:
next natNumsEnum

In [ ]:
next natNumsEnum

The function `printFirst` takes two arguments:
- `s` is a sequence of integers,
- `n` is a natural number.

It prints the first `n` numbers of the sequence `s`.  The function `Seq.take n s` returns the sequence of the first `n` elements of `s`.

In [ ]:
let printFirst (s: seq<int>) (n: int) : unit =
    for x in Seq.take n s do
        printfn "%d" x

In [ ]:
printFirst (countFrom 7) 3

The function `filterStream` takes two arguments:
  * `s` is a sequence of integers,
  * `p` is a predicate, i.e. a function that maps integers to truth values.

It returns a new sequence that contains only those values `x` from `s`
such that `p x` is `true`.
$$ \texttt{filterStream}(S, p) = [ x \in S \mid p(x) ] $$
The F$^\sharp$ library provides the same functionality as `Seq.filter p s`.  We define `filterStream` nevertheless in order to show how a `for` loop and an `if` are used inside a sequence expression.

In [ ]:
let filterStream (s: seq<int>) (p: int -> bool) : seq<int> =
    seq { for x in s do
              if p x then
                  yield x }

Let us test `filterStream` by creating a stream of numbers divisible by $3$.

In [ ]:
let divBy3 = filterStream (countFrom 1) (fun n -> n % 3 = 0)
printFirst divBy3 5

The function `filterPrime` takes a sequence `s` of numbers, yields the first element `p`, filters all subsequent multiples of `p`, and calls `filterPrime` recursively on the resulting sequence.
$$ \texttt{filterPrime}(p :: R) =
  p :: \texttt{filterPrime}\bigl([ x \in R \mid x \,\texttt{\%}\, p \not= 0]\bigr) $$
Inside a sequence expression, `yield! s` produces all elements of the sequence `s`.  This is the counterpart of `yield*` in TypeScript.  The function `Seq.head` returns the first element of a sequence, while `Seq.tail` returns the sequence of the remaining elements.  As sequences are lazy, the recursive call of `filterPrime` is only evaluated when the second element of the result is needed.  Therefore, the recursion does not run forever.

As sequences have no state, both `Seq.head s` and `Seq.tail s` traverse `s` from its beginning.  Hence, the numbers are filtered several times.  For the first hundred prime numbers, this is still fast.

In [ ]:
let rec filterPrime (s: seq<int>) : seq<int> =
    seq { let p = Seq.head s   // get first number
          yield p              // and return it, since it is prime
          // remove all multiples of prime p
          yield! filterPrime (filterStream (Seq.tail s) (fun x -> x % p <> 0)) }

The value `primes` is the infinite sequence of *prime numbers*, which is generated using
the [*Sieve of Eratosthenes*](https://en.wikipedia.org/wiki/Sieve_of_Eratosthenes).  As no element of a sequence is computed before it is needed, `primes` can be defined as a value: Unlike in TypeScript, we need no function to create the sequence.

In [ ]:
let primes : seq<int> = filterPrime (countFrom 2)

In [ ]:
printFirst primes 100